# 1. Why Does the Kernel Need an MMU?

The kernel is privileged and can control memory directly. Why bother enabling the MMU?

**The MMU is not just about protecting the kernel from user programs. It also helps the kernel protect itself from bugs and manage memory more flexibly.**

## Benefit 1: Kernel Stack Guard Pages

Each process has a kernel stack, used for function calls, local variables, and saved state.

Without guard pages:

```text
High Address
+---------------------+
| Kernel Stack A      |
|                     |
+---------------------+
| Kernel Stack B      |
|                     |
+---------------------+
Low Address
```

If Stack A overflows downward, it may overwrite another stack or unrelated kernel data.

**Solution: Insert an unmapped virtual page below each kernel stack.**

```text
High Address
+---------------------+
| Kernel Stack A      |
|                     |
+---------------------+
| GUARD PAGE          |  ← Unmapped
+---------------------+
| Kernel Stack B      |
|                     |
+---------------------+
Low Address
```

When the stack overflows into the guard page:

1. CPU tries to access the address.
2. MMU finds no valid mapping.
3. A page fault occurs.
4. xv6 typically panics instead of silently corrupting memory.

A guard page does not need physical RAM because it is intentionally unmapped.

**Key insight: The MMU helps the kernel detect its own mistakes.**

## Benefit 2: Memory Access Permissions

The kernel can accidentally corrupt itself.

For example, xv6 maps kernel code with:

```c
PTE_R | PTE_X
```

This means:

- Read allowed.
- Execute allowed.
- Write forbidden.

If buggy kernel code accidentally writes to its code region, the MMU raises an exception.

But isn't the kernel the boss? Can't it modify its own page table?

**Yes! The kernel can change its own permissions.**

The goal is not to stop a privileged kernel from deliberately changing memory. It is to catch accidental mistakes.

**Key insight: Being privileged doesn't mean every memory operation should automatically succeed.**

## Benefit 3: Flexible Address Mapping

Without virtual memory, addresses directly refer to physical locations.

With an MMU, the kernel can organize its virtual address space independently of physical RAM.

For example:

```text
Kernel Virtual Memory         Physical RAM

Kernel Stack A  ------------> Physical Page X
Guard Page                   (No mapping)
Kernel Stack B  ------------> Physical Page Y
```

Physical Pages X and Y do not need to be adjacent.

This makes it possible to arrange kernel stacks with guard pages even when the actual physical pages are scattered.

In xv6, most kernel memory uses **identity mapping**:

```text
Virtual Address = Physical Address
```

Therefore, xv6 uses relatively little of the MMU's mapping flexibility.

## Trade-offs

| MMU Enabled | MMU Disabled |
|---|---|
| Guard pages | No page-table guard pages |
| Memory permissions | No page-table permission checks |
| Flexible address mappings | Direct physical addressing |
| Page table and TLB overhead | Simpler address translation |
| More complex transitions | Simpler addressing model |

**Conclusion:** The kernel can run without paging on suitable hardware, but keeping the MMU enabled provides useful protection and flexibility. It is an engineering trade-off.

---

# 2. Trampoline: Is It Just Switching `satp`?

**Switching `satp` is the core operation, but the trampoline must also prepare the CPU to safely transition between user and kernel execution environments.**

## Step 1: What Happens During a System Call?

Suppose a user program executes:

```asm
ecall
```

The RISC-V hardware:

1. Switches privilege from User Mode to Supervisor Mode.
2. Saves the interrupted program counter in `sepc`.
3. Transfers execution to the trap handler specified by `stvec`.

But the hardware does **not** automatically:

- Switch the page table.
- Switch to the kernel stack.
- Save every general-purpose register.

Immediately after the trap:

```text
Privilege = Supervisor Mode
satp      = User Page Table
sp        = User Stack Pointer
Registers = User Register Values
```

The CPU is privileged now, but it is still using much of the user's execution environment.

## Step 2: Why Can't We Immediately Execute Kernel C Code?

Normal C functions need a valid stack and registers that they can safely use.

But the CPU is still using the user stack pointer and user page table.

The kernel must first establish a safe execution environment.

## Step 3: Enter the Trampoline

In xv6, `kernel/trampoline.S` contains assembly code for this transition.

Two important entry points:

- `uservec`: User → Kernel
- `userret`: Kernel → User

Simplified process:

```text
User Program
     |
     | ecall
     v
Trampoline: uservec
     |
     | Save user registers to trapframe
     | Load kernel stack pointer
     | Load kernel page table information
     | Switch satp
     v
Kernel: usertrap()
     |
     | Handle syscall
     v
Trampoline: userret
     |
     | Switch back to user page table
     | Restore user registers
     | sret
     v
User Program
```

The trapframe is a memory region used to save the user's register state.

## Step 4: Is Switching `satp` Complicated?

The actual page table switch takes only one instruction:

```asm
csrw satp, t1
```

Here, `t1` holds the new `satp` value, including the root page table's physical page number.

xv6 also uses:

```asm
sfence.vma zero, zero
```

to synchronize address translation state.

**The difficult part is not changing `satp`. It is ensuring the CPU can continue executing correctly after the change.**

## Step 5: Why Must the Trampoline Be Special?

The trampoline must be mapped at the **same virtual address** in both the user and kernel page tables.

```text
User Page Table
    |
    | VA 0x3FFFFFF000
    v
Physical Trampoline Page
    ^
    | VA 0x3FFFFFF000
    |
Kernel Page Table
```

This is achieved by adding mappings to both page tables that point to the same physical page.

When `satp` changes, the CPU's current virtual program counter still refers to valid trampoline code.

Without this arrangement, switching page tables could make the next instruction inaccessible.

The trampoline is therefore a shared transition region between two virtual address spaces.

## Step 6: Is the Trampoline a Context Switch?

They share a similar idea: saving state, changing execution context, and restoring state.

However, they serve different purposes.

| | Trampoline | `swtch()` |
|---|---|---|
| Purpose | User ↔ Kernel transition | Switch kernel execution contexts |
| Saves registers | User registers in trapframe | Kernel context registers |
| Switches `satp` | Yes | Not directly in xv6 |
| Chooses another process | No | Scheduler makes that decision |
| Source | `trampoline.S` | `swtch.S` |

A syscall often follows:

```text
Process A (User)
        ↓
Trampoline
        ↓
Kernel
        ↓
Trampoline
        ↓
Process A (User)
```

But if the kernel blocks or yields, the scheduler may run another process before Process A resumes.

**Key insight: Entering the kernel does not necessarily mean switching processes.**